In [1]:
import os
import time

import re
import torch
import pandas as pd
import jsonlines
from tqdm import tqdm
from dotenv import load_dotenv

import google.generativeai as genai
from openai import OpenAI
from unsloth import FastLanguageModel

from sklearn.metrics import classification_report, f1_score

c:\Users\chis\.conda\envs\support-ticket-router\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\chis\AppData\Local\Temp\ipykernel_20220\1170246599.py:11: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W0416 06:38:27.149000 20220 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [79]:
load_dotenv()
gpt_client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)
genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))
gemma4_model = genai.GenerativeModel("gemma-4-31b-it")

In [80]:
LABELS = [
    "api",
    "billing",
    "cancellation",
    "complaint",
    "technical",
    "upgrade"
]

In [81]:
with jsonlines.open("../data/test.jsonl") as reader:
    test_data = list(reader)

test_df = pd.DataFrame(test_data)

In [82]:
def format_prompt(text: str) -> str:
    prompt = f"""Classify this customer support message into one of these intents.
    Intents: {', '.join(LABELS)}
    Message: {text}
    Intent:"""
    return prompt

def extract_label(raw: str) -> str:
    raw = re.sub(r'<thought>.*?</thought>', '', raw, flags=re.DOTALL)
    raw = re.sub(r'<think>.*?</think>',   '', raw, flags=re.DOTALL)
    raw = raw.strip().lower()
    
    words = re.findall(r"\b(api|billing|cancellation|complaint|technical|upgrade)\b", raw)

    if words:
        return words[-1]
    return "unknown"

In [6]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Phi-4-mini-instruct-bnb-4bit",
    max_seq_length=512,
    dtype=None,
    load_in_4bit=True
)

model.load_adapter("../models/phi4-intent-finetuned")

FastLanguageModel.for_inference(model)

This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.


==((====))==  Unsloth 2026.4.2: Fast Phi3 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 3060 Laptop GPU. Num GPUs = 1. Max memory: 6.0 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.10.0+cu130. CUDA: 8.6. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████| 64/64 [00:00<00:00, 2285.74it/s]


Phi3ForCausalLM(
  (model): Phi3Model(
    (embed_tokens): Embedding(200064, 3072, padding_idx=200029)
    (layers): ModuleList(
      (0-31): 32 x Phi3DecoderLayer(
        (self_attn): Phi3Attention(
          (o_proj): lora.Linear4bit(
            (base_layer): Linear4bit(in_features=3072, out_features=3072, bias=False)
            (lora_dropout): ModuleDict(
              (default): Dropout(p=0.05, inplace=False)
            )
            (lora_A): ModuleDict(
              (default): Linear(in_features=3072, out_features=16, bias=False)
            )
            (lora_B): ModuleDict(
              (default): Linear(in_features=16, out_features=3072, bias=False)
            )
            (lora_embedding_A): ParameterDict()
            (lora_embedding_B): ParameterDict()
            (lora_magnitude_vector): ModuleDict()
          )
          (qkv_proj): Linear4bit(in_features=3072, out_features=5120, bias=False)
        )
        (mlp): Phi3MLP(
          (gate_up_proj): Linear4bit(

In [83]:
def classify_phi4(text: str) -> tuple[str, float]:
    start = time.time()
    
    inputs = tokenizer(
        format_prompt(text),
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=3,
        temperature=0.1,
        do_sample=False
    )

    result = tokenizer.decode(outputs[0], skip_special_tokens=True)

    raw = result.split()[-1].strip().split()[0].lower()
    predicted = extract_label(raw)

    latency = (time.time() - start) * 1000 

    return predicted, latency


In [87]:
def classify_gemma_4(text: str) -> tuple[str, float]:
    start = time.time()
    
    response = gemma4_model.generate_content(
        format_prompt(text),
        generation_config={
            "temperature": 0.1,
            "max_output_tokens": 10
        }
        
    )
    
    latency = (time.time() - start) * 1000

    raw = response.text
    predicted = extract_label(raw)

    return predicted, latency

In [85]:
def classify_gpt(text: str) -> tuple[str, float]:
    start = time.time()
    response = gpt_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": format_prompt(text)}],
        temperature=0.1,
        max_tokens=10
    )
    latency   = (time.time() - start) * 1000
    raw       = response.choices[0].message.content
    predicted = extract_label(raw)
    return predicted, latency

In [86]:
print("Running SLM predictions")
slm_preds = []
slm_latencies = []
for text in tqdm(test_df["text"].tolist()):
    start = time.time()
    pred, lat = classify_phi4(text)
    slm_preds.append(pred)
    slm_latencies.append(lat)

test_df["slm_pred"]    = slm_preds
test_df["slm_latency"] = slm_latencies

Running SLM predictions


100%|██████████| 150/150 [06:05<00:00,  2.44s/it]


In [88]:
test_df

,text,label,slm_pred,slm_latency
0,I expected much better from a company like yours.,complaint,complaint,2939.860582
1,There is a discrepancy in how the filters are ...,technical,technical,2877.956629
2,I'd like to terminate my plan. Could you let m...,cancellation,cancellation,2676.025391
3,This is ridiculous.,complaint,complaint,2601.274729
4,Where can I regenerate my secret key?,api,api,2383.010864
...,...,...,...,...
145,I want a cheaper plan. Now.,upgrade,upgrade,2311.091661
146,Seriously? I'm still waiting for a real person...,complaint,complaint,2283.192873
147,"I don't know what's going on over there, but t...",complaint,complaint,2290.770769
148,I keep hitting the rate limit every 5 minutes....,api,api,2263.117790


In [89]:
print("Running GEMMA 4 predictions")
gemma4_preds     = []
gemma4_latencies = []
for text in tqdm(test_df["text"].tolist()):
    pred, lat = classify_gemma_4(text)
    gemma4_preds.append(pred)
    gemma4_latencies.append(lat)
    time.sleep(1)

test_df["gemma4_pred"] = gemma4_preds
test_df["gemma4_latency"] = gemma4_latencies

Running GEMMA 4 predictions


100%|██████████| 150/150 [24:55<00:00,  9.97s/it]


In [90]:
test_df

,text,label,slm_pred,slm_latency,gemma4_pred,gemma4_latency
0,I expected much better from a company like yours.,complaint,complaint,2939.860582,complaint,6610.695601
1,There is a discrepancy in how the filters are ...,technical,technical,2877.956629,technical,8336.368084
2,I'd like to terminate my plan. Could you let m...,cancellation,cancellation,2676.025391,cancellation,7554.374933
3,This is ridiculous.,complaint,complaint,2601.274729,complaint,6467.719793
4,Where can I regenerate my secret key?,api,api,2383.010864,api,7346.113682
...,...,...,...,...,...,...
145,I want a cheaper plan. Now.,upgrade,upgrade,2311.091661,billing,8875.776768
146,Seriously? I'm still waiting for a real person...,complaint,complaint,2283.192873,complaint,8140.782833
147,"I don't know what's going on over there, but t...",complaint,complaint,2290.770769,complaint,8837.022781
148,I keep hitting the rate limit every 5 minutes....,api,api,2263.117790,api,9835.876703


In [91]:
print("Running GPT-4o-mini predictions")
gpt_preds     = []
gpt_latencies = []

for text in tqdm(test_df["text"].tolist()):
    pred, lat = classify_gpt(text)
    gpt_preds.append(pred)
    gpt_latencies.append(lat)
    time.sleep(0.5)

test_df["gpt_pred"] = gpt_preds
test_df["gpt_latency"] = gpt_latencies

Running GPT-4o-mini predictions


100%|██████████| 150/150 [03:08<00:00,  1.26s/it]


In [92]:
test_df

,text,label,slm_pred,slm_latency,gemma4_pred,gemma4_latency,gpt_pred,gpt_latency
0,I expected much better from a company like yours.,complaint,complaint,2939.860582,complaint,6610.695601,complaint,1934.673071
1,There is a discrepancy in how the filters are ...,technical,technical,2877.956629,technical,8336.368084,technical,1294.438601
2,I'd like to terminate my plan. Could you let m...,cancellation,cancellation,2676.025391,cancellation,7554.374933,cancellation,851.923704
3,This is ridiculous.,complaint,complaint,2601.274729,complaint,6467.719793,complaint,750.247478
4,Where can I regenerate my secret key?,api,api,2383.010864,api,7346.113682,technical,732.103586
...,...,...,...,...,...,...,...,...
145,I want a cheaper plan. Now.,upgrade,upgrade,2311.091661,billing,8875.776768,upgrade,651.013136
146,Seriously? I'm still waiting for a real person...,complaint,complaint,2283.192873,complaint,8140.782833,complaint,2457.003832
147,"I don't know what's going on over there, but t...",complaint,complaint,2290.770769,complaint,8837.022781,complaint,628.313541
148,I keep hitting the rate limit every 5 minutes....,api,api,2263.117790,api,9835.876703,api,650.395155


In [97]:
print("SLM (Fine-tuned Phi-4-mini)")
print(classification_report(
    test_df["label"],
    test_df["slm_pred"],
    labels=LABELS,
    digits=4
))

SLM (Fine-tuned Phi-4-mini)
              precision    recall  f1-score   support

         api     1.0000    1.0000    1.0000        25
     billing     1.0000    0.9600    0.9796        25
cancellation     1.0000    0.9600    0.9796        25
   complaint     0.8929    1.0000    0.9434        25
   technical     0.9583    0.9200    0.9388        25
     upgrade     1.0000    1.0000    1.0000        25

    accuracy                         0.9733       150
   macro avg     0.9752    0.9733    0.9736       150
weighted avg     0.9752    0.9733    0.9736       150



In [95]:
print("Baseline (Gemma 4 31B prompt-based)")
print(classification_report(
    test_df["label"],
    test_df["gemma4_pred"],
    labels=LABELS,
    digits=4
))

Baseline (Gemma 4 31B prompt-based)
              precision    recall  f1-score   support

         api     0.9600    0.9600    0.9600        25
     billing     0.7812    1.0000    0.8772        25
cancellation     0.9615    1.0000    0.9804        25
   complaint     0.9259    1.0000    0.9615        25
   technical     0.9200    0.9200    0.9200        25
     upgrade     1.0000    0.6000    0.7500        25

    accuracy                         0.9133       150
   macro avg     0.9248    0.9133    0.9082       150
weighted avg     0.9248    0.9133    0.9082       150



In [96]:
print("Baseline (GPT-4o-mini prompt-based)")
print(classification_report(
    test_df["label"],
    test_df["gpt_pred"],
    labels=LABELS,
    digits=4
))

Baseline (GPT-4o-mini prompt-based)
              precision    recall  f1-score   support

         api     0.9167    0.4400    0.5946        25
     billing     0.9200    0.9200    0.9200        25
cancellation     0.8929    1.0000    0.9434        25
   complaint     0.8929    1.0000    0.9434        25
   technical     0.6053    0.9200    0.7302        25
     upgrade     1.0000    0.7600    0.8636        25

    accuracy                         0.8400       150
   macro avg     0.8713    0.8400    0.8325       150
weighted avg     0.8713    0.8400    0.8325       150



In [ ]:
slm_f1 = f1_score(test_df["label"], test_df["slm_pred"], average="macro")
gemma_f1 = f1_score(test_df["label"], test_df["gemma4_pred"], average="macro")
gpt_f1    = f1_score(test_df["label"], test_df["gpt_pred"],   average="macro")

slm_avg_lat = sum(slm_latencies) / len(slm_latencies)
gemma_avg_lat = sum(gemma4_latencies) / len(gemma4_latencies)
gpt_avg_lat   = sum(gpt_latencies) / len(gpt_latencies)

print(f"{'Metric':<30} {'SLM (Phi4 fine-tuned)':<30} {'Gemma 4 31B':<30} {'GPT-4o-mini':<30}")
print(f"{'F1 Macro':<30} {slm_f1*100:<30.2f} {gemma_f1*100:<30.2f} {gpt_f1*100:<30.2f}")
print(f"{'Avg latency (ms)':<30} {slm_avg_lat:<30.0f} {gemma_avg_lat:<30.0f} {gpt_avg_lat:<30.0f}")
print(f"{'Cost/call':<30} {'~$0 (local)':<30} {'$0 (free tier)':<30} {'$0.000015 (gpt-4o-mini)'}")
print(f"{'Deployment':<30} {'Local (RTX 3060 6Gb)':<30} {'GOOGLE API':<30} {'OpenAI API'}")
print("_" * 120)
print(f"\nVs Gemma 4 31B : SLM is{gemma_avg_lat/slm_avg_lat:.0f}x faster")
print(f"Vs GPT-4o-mini : SLM is {gpt_avg_lat/slm_avg_lat:.0f}x faster")

Metric                         SLM (Phi4 fine-tuned)          Gemma 4 31B                    GPT-4o-mini                   
F1 Macro                       97.36                          90.82                          83.25                         
Avg latency (ms)               2434                           8972                           758                           
Cost/call                      ~$0 (local)                    $0 (free tier)                 $0.000015 (gpt-4o-mini)
Deployment                     Local (RTX 3060)               GOOGLE API                     OpenAI API
________________________________________________________________________________________________________________________

Vs Gemma 4 31B : SLM is4x faster
Vs GPT-4o-mini : SLM is 0x faster


In [117]:
test_df.to_csv("../data/eval_results.csv", index=False)